# DPO Дообучение + Оценка

**План:**
1. Подготовка DPO датасета
2. DPO дообучение поверх SFT чекпоинта (beta=0.1) - 2 эпохи
3. Эксперимент с beta ∈ {0.01, 0.1, 0.5}
4. Сравнение трёх версий: BASE / SFT / DPO
5. Оценка через логиты (модель все-таки обучена давать развернутые ответы, поэтому пробуем логиты)



In [ ]:
!pip install -q trl transformers datasets peft bitsandbytes accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.5 MB/s eta 0:00:00


In [ ]:
import json
import gc
import os
import shutil
from pathlib import Path
from typing import Optional

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import DPOTrainer, DPOConfig
from tqdm.auto import tqdm

print('Импорты загружены')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет"}')

Импорты загружены
GPU: нет


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Модели
BASE_MODEL_ID  = "google/gemma-3-4b-it"
SFT_CHECKPOINT = "/content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2"

# Пути
DPO_DATA_PATH  = "/content/drive/MyDrive/Colab Notebooks/dpo_natural_science_500_hard.json"
DPO_OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/gemma3_dpo"
RESULTS_DIR    = "/content/drive/MyDrive/mera_results_dpo"
os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# DPO гиперпараметры
BETA_VALUES       = [0.01, 0.1, 0.5]   # эксперимент с beta
BETA_DEFAULT      = 0.1                # основной запуск
DPO_EPOCHS        = 2                  # 2 эпохи (500 примеров)
DPO_LR            = 5e-5
DPO_BATCH         = 1
DPO_GRAD_ACCUM    = 8
DPO_MAX_LENGTH    = 512

# Оценка
MAX_EVAL_SAMPLES  = 1000
NUM_FEWSHOT       = 5
ANSWER_LETTERS    = ["A", "B", "C", "D"]

print('Конфигурация загружена')
print(f'SFT чекпоинт: {SFT_CHECKPOINT}')

Конфигурация загружена
SFT чекпоинт: /content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2


## 1. Подготовка DPO датасета

In [ ]:
with open(DPO_DATA_PATH) as f:
    raw_data = json.load(f)

#если не добавить пробел, токенизатор сходит с ума,т.к. в ответе одна буква только
def fix_whitespace(item):
    return {
        "prompt":   item["prompt"],
        "chosen":   " " + item["chosen"].strip(),
        "rejected": " " + item["rejected"].strip(),
    }

raw_data_fixed = [fix_whitespace(item) for item in raw_data]
dpo_dataset = Dataset.from_list(raw_data_fixed)

# Разбивка train/test
dpo_dataset = dpo_dataset.train_test_split(test_size=0.1, seed=42)

dpo_train = dpo_dataset["train"]
dpo_test  = dpo_dataset["test"]

print(f"Train: {len(dpo_train)}, Test: {len(dpo_test)}")
print(f"\nПример:")
print(dpo_train[0])

Train: 450, Test: 50

Пример:
{'prompt': 'Пара приходит на предварительную генетическую консультацию, потому что у них обоих в семейном анамнезе α-талассемия. У женщины минимально снижена концентрация гемоглобина. Генетические исследования показывают делецию одного гена. У мужчины микроцитарная анемия и делеция двух генов. Если делеция двух генов находится в трансе (одна делеция в гене матери и одна делеция в гене отца), какой из следующих процентов их потомства будет иметь делецию двух генов?\nA) 0%\nB) 25%\nC) 50%\nD) 75%', 'chosen': ' C', 'rejected': ' B'}


## 2. Загрузка SFT модели

In [ ]:
def load_model_for_dpo(checkpoint_path: str):
    """Загружает модель в 4bit для DPO обучения"""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

    model = AutoModelForCausalLM.from_pretrained(
        checkpoint_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False

    return model, tokenizer


def add_lora_for_dpo(model):
    """Добавляет LoRA адаптер для DPO"""
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    return get_peft_model(model, lora_config)

## 3. DPO обучение (основной запуск beta=0.1)

In [ ]:
# Функция DPO обучения
def run_dpo(
    sft_checkpoint: str,
    train_dataset: Dataset,
    beta: float = 0.1,
    num_epochs: int = 1,
    output_suffix: str = "",
) -> str:
    """
    Запускает DPO обучение поверх SFT чекпоинта.
    Возвращает путь к сохранённому чекпоинту.
    """
    print(f'DPO обучение: beta={beta}, epochs={num_epochs}')
    print()

    output_dir = f"{DPO_OUTPUT_DIR}/beta_{str(beta).replace('.', '_')}{output_suffix}"
    tmp_dir    = f"/content/tmp_dpo_beta_{str(beta).replace('.', '_')}"

    model, tokenizer = load_model_for_dpo(sft_checkpoint)
    model = add_lora_for_dpo(model)

    dpo_config = DPOConfig(
        output_dir                   = tmp_dir,
        num_train_epochs             = num_epochs,
        per_device_train_batch_size  = DPO_BATCH,
        gradient_accumulation_steps  = DPO_GRAD_ACCUM,
        learning_rate                = DPO_LR,
        beta                         = beta,
        bf16                         = True,
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs= {"use_reentrant": False},
        logging_steps                = 5,
        save_strategy                = "no",
        eval_strategy                = "no",
        report_to                    = "none",
        optim                        = "adamw_torch_fused",
        remove_unused_columns        = False,
        seed                         = 42,
    )

    trainer = DPOTrainer(
        model            = model,
        args             = dpo_config,
        train_dataset    = train_dataset,
        processing_class = tokenizer,
    )

    trainer.train()

    # Сохраняем
    print(f'Сохраняем чекпоинт: {output_dir}')
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Чистим память
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    if Path(tmp_dir).exists():
        shutil.rmtree(tmp_dir, ignore_errors=True)

    print(f'DPO чекпоинт сохранён: {output_dir}')
    return output_dir

In [ ]:
#Основной запуск DPO (beta=0.1)
dpo_main_path = run_dpo(
    sft_checkpoint = SFT_CHECKPOINT,
    train_dataset  = dpo_train,
    beta           = 0.1,
    num_epochs     = DPO_EPOCHS,
)
print(f'Основной DPO чекпоинт: {dpo_main_path}')

DPO обучение: beta=0.1, epochs=2



Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Step,Training Loss
5,0.685701
10,0.672716
15,0.653955
20,0.628085
25,0.600298
30,0.593568
35,0.635951
40,0.531529
45,0.530043
50,0.506347


Сохраняем чекпоинт: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1
DPO чекпоинт сохранён: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1
Основной DPO чекпоинт: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1


## 4. Эксперимент с beta ∈ {0.01, 0.1, 0.5}

In [ ]:
#Запуск для всех beta
#Запускаем по одному
#beta=0.1 уже обучен выше, можно пропустить

beta_checkpoints = {
    0.1: dpo_main_path,
}

for beta in [0.01, 0.5]:  # остальные два
    path = run_dpo(
        sft_checkpoint = SFT_CHECKPOINT,
        train_dataset  = dpo_train,
        beta           = beta,
        num_epochs     = DPO_EPOCHS,
    )
    beta_checkpoints[beta] = path

print('\nВсе beta обучены:')
for beta, path in sorted(beta_checkpoints.items()):
    print(f'  beta={beta}: {path}')

DPO обучение: beta=0.01, epochs=2



Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

...

Adding EOS to train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Step,Training Loss
5,0.657208
10,0.612075
15,0.576385
20,0.572504
25,0.517036
30,0.531248
35,0.560909
40,0.500492
45,0.518045
50,0.484647


Step,Training Loss
5,0.657208
10,0.612075
15,0.576385
20,0.572504
25,0.517036
30,0.531248
35,0.560909
40,0.500492
45,0.518045
50,0.484647


Сохраняем чекпоинт: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_5
DPO чекпоинт сохранён: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_5

Все beta обучены:
  beta=0.01: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_01
  beta=0.1: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1
  beta=0.5: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_5


## 5. Оценка на ruMMLU

In [ ]:
mera = load_dataset("ai-forever/MERA", name="rummlu")
public_df = mera["public_test"].to_pandas()

# domain спрятан в meta, достаём его, чтобы отобрать только естественные науки
public_df["domain"] = public_df["meta"].apply(
    lambda m: m.get("domain") if isinstance(m, dict) else None
)


SCIENCE_DOMAINS = [
    "anatomy", "astronomy", "college_biology", "college_chemistry",
    "college_medicine", "college_physics", "conceptual_physics",
    "high_school_biology", "high_school_chemistry", "high_school_physics",
    "medical_genetics", "nutrition", "professional_medicine", "virology",
    "high_school_geography",
]

def filter_natural_sciences(df):
    return df[df["domain"].isin(SCIENCE_DOMAINS)].reset_index(drop=True)

print(f'Всего примеров в ruMMLU: {len(public_df)}')
print(f'Естественных наук: {len(filter_natural_sciences(public_df))}')

Всего примеров в ruMMLU: 10033
Естественных наук: 2033


In [ ]:
with open(DPO_DATA_PATH) as f:
    raw_data = json.load(f)

dpo_prompts = set(item["prompt"] for item in raw_data)
print(f"DPO промптов: {len(dpo_prompts)}")

# Фильтруем ruMMLU, т.е. убираем те что были в обучении
def format_prompt_for_matching(row):
    inputs = row["inputs"] if isinstance(row["inputs"], dict) else {}
    return (
        f"{inputs.get('text', '')}\n"
        f"A) {inputs.get('option_a', '')}\n"
        f"B) {inputs.get('option_b', '')}\n"
        f"C) {inputs.get('option_c', '')}\n"
        f"D) {inputs.get('option_d', '')}"
    )

public_df["_prompt"] = public_df.apply(format_prompt_for_matching, axis=1)
clean_eval_df = public_df[~public_df["_prompt"].isin(dpo_prompts)].drop(columns=["_prompt"])

print(f"Всего в ruMMLU:         {len(public_df)}")
print(f"Было в DPO обучении:    {len(public_df) - len(clean_eval_df)}")
print(f"Чистых для оценки:      {len(clean_eval_df)}")

DPO промптов: 500
Всего в ruMMLU:         10033
Было в DPO обучении:    500
Чистых для оценки:      9533


In [ ]:
#Функции оценки
def build_eval_prompt(row: pd.Series, fewshot_pool: pd.DataFrame) -> str:
    """Строит few-shot промпт для оценки"""
    def format_example(r, with_answer=True):
        instruction = r["instruction"]
        prompt = instruction.format(
            subject=r.get("subject", ""),
            text=r.get("text", ""),
            option_a=r.get("option_a", ""),
            option_b=r.get("option_b", ""),
            option_c=r.get("option_c", ""),
            option_d=r.get("option_d", ""),
        )
        if with_answer:
            prompt += f" {str(r.get('outputs', '')).strip()}"
        return prompt

    domain     = row.get("domain", "")
    current_id = row["meta"].get("id") if isinstance(row.get("meta"), dict) else None
    pool = fewshot_pool[fewshot_pool["domain"] == domain]
    if current_id is not None:
        pool = pool[pool["meta"].apply(
            lambda m: m.get("id") if isinstance(m, dict) else None
        ) != current_id]

    shots = pool.sample(min(NUM_FEWSHOT, len(pool)), random_state=42)
    prompt = ""
    for _, shot in shots.iterrows():
        prompt += format_example(shot, with_answer=True) + "\n\n"
    prompt += format_example(row, with_answer=False)
    return prompt



#Оценка через логиты
def evaluate_with_logits(
    model,
    tokenizer,
    eval_df: pd.DataFrame,
    fewshot_pool: pd.DataFrame,
    label: str = "model",
) -> pd.DataFrame:
    """
    Оценка через логиты честнее для модели обученной на развёрнутых ответах.
    Смотрим на вероятности токенов A/B/C/D напрямую.
    """
    # Токены для A B C D (латиница + кириллица)
    token_ids = {}
    for letter, cyrillic in [("A","А"), ("B","В"), ("C","С"), ("D","Д")]:
        ids_lat = tokenizer.encode(letter, add_special_tokens=False)
        ids_cyr = tokenizer.encode(cyrillic, add_special_tokens=False)
        token_ids[letter] = ids_lat + ids_cyr

    results = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Eval [{label}]"):
        prompt = build_eval_prompt(row, fewshot_pool)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

        with torch.no_grad():
            logits = model(**inputs).logits[0, -1, :]

        scores    = {letter: max(logits[tid].item() for tid in tids)
                     for letter, tids in token_ids.items()}
        predicted = max(scores, key=scores.get)
        gold      = str(row.get("outputs", "")).strip().upper()
        meta_id   = row["meta"].get("id") if isinstance(row.get("meta"), dict) else None

        results.append({
            "id":        meta_id,
            "domain":    row.get("domain", ""),
            "subject":   row.get("subject", ""),
            "gold":      gold,
            "predicted": predicted,
            "correct":   predicted == gold,
            "scores":    str(scores),
        })

    return pd.DataFrame(results)




def print_results(df: pd.DataFrame, label: str = ""):
    acc = df["correct"].mean()
    print(f"\n{'='*55}")
    print(f"Результаты: {label}")
    print(f"{'='*55}")
    print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)  |  n={len(df)}")
    by_domain = (
        df.groupby("domain")["correct"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "accuracy", "count": "n"})
        .sort_values("accuracy", ascending=False)
    )
    print(by_domain.to_string(float_format=lambda x: f"{x:.3f}"))
    return acc





def load_model_for_eval(model_path: str, label: str = ""):
    """Загружает модель для оценки (4bit)"""
    print(f"\nЗагружаем [{label}]: {model_path}")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()
    return model, tokenizer





def generate_answer(model, tokenizer, prompt: str, max_new_tokens: int = 5) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
#MAX_EVAL_SAMPLES = 100

In [ ]:
#Проведем первичную оценку, чтобы посмотреть, как модель отвечает
model_dpo, tok_dpo = load_model_for_eval(dpo_main_path, label="DPO beta=0.5")

eval_df = clean_eval_df.sample(
    min(MAX_EVAL_SAMPLES, len(clean_eval_df)), random_state=42
).reset_index(drop=True)
eval_sci = filter_natural_sciences(eval_df)

print(f"Всего для оценки:  {len(eval_df)}")
print(f"Естественных наук: {len(eval_sci)}")

for _, row in eval_df.head(20).iterrows():
    prompt = build_eval_prompt(row, clean_eval_df)
    answer = generate_answer(model_dpo, tok_dpo, prompt, max_new_tokens=5)
    print(f"gold={row['outputs']}  generated={repr(answer.strip())}")


Загружаем [DPO beta=0.5]: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Всего для оценки:  100
Естественных наук: 18
gold=B  generated='B'
gold=D  generated='C'
gold=A  generated='B'
gold=D  generated='B'
gold=D  generated='C'
gold=A  generated='B'
gold=B  generated='C'
gold=B  generated='B'
gold=C  generated='C'
gold=C  generated='C'
gold=C  generated='C'
gold=D  generated='B'
gold=D  generated='C'
gold=D  generated='C'
gold=A  generated='C'
gold=A  generated='C'
gold=B  generated='C'
gold=A  generated='B'
gold=C  generated='C'
gold=C  generated='B'


Она отвечает буквами, но только C и D... Печально

In [ ]:
# На всякий случай сравним 2-умя методами
gen_correct = 0
logit_correct = 0

token_ids = {}
for letter, cyrillic in [("A","А"), ("B","В"), ("C","С"), ("D","Д")]:
    ids_lat = tok_dpo.encode(letter, add_special_tokens=False)
    ids_cyr = tok_dpo.encode(cyrillic, add_special_tokens=False)
    token_ids[letter] = ids_lat + ids_cyr

for _, row in eval_df.head(100).iterrows():
    prompt = build_eval_prompt(row, clean_eval_df)
    inputs = tok_dpo(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model_dpo.device)
    gold = str(row["outputs"]).strip().upper()

    # Генерация
    with torch.no_grad():
        out = model_dpo.generate(**inputs, max_new_tokens=3, do_sample=False,
                                  pad_token_id=tok_dpo.eos_token_id)
    gen_pred = tok_dpo.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()[0].upper()
    gen_correct += (gen_pred == gold)

    # Логиты
    with torch.no_grad():
        logits = model_dpo(**inputs).logits[0, -1, :]
    scores = {l: max(logits[tid].item() for tid in tids) for l, tids in token_ids.items()}
    logit_pred = max(scores, key=scores.get)
    logit_correct += (logit_pred == gold)

print(f"Генерация: {gen_correct/100:.1%}")
print(f"Логиты:    {logit_correct/100:.1%}")

Генерация: 28.0%
Логиты:    29.0%


In [ ]:
# Ну ладно, очень низко, проверим BASE тогда
model_base, tok_base = load_model_for_eval(BASE_MODEL_ID, label="BASE")

base_correct = 0
for _, row in eval_df.head(100).iterrows():
    prompt = build_eval_prompt(row, clean_eval_df)
    inputs = tok_base(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model_base.device)
    gold = str(row["outputs"]).strip().upper()
    with torch.no_grad():
        logits = model_base(**inputs).logits[0, -1, :]
    scores = {l: max(logits[tid].item() for tid in tids) for l, tids in token_ids.items()}
    pred = max(scores, key=scores.get)
    base_correct += (pred == gold)

print(f"BASE на тех же 100: {base_correct/100:.1%}")
del model_base, tok_base
gc.collect(); torch.cuda.empty_cache()


Загружаем [BASE]: google/gemma-3-4b-it


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

BASE на тех же 100: 27.0%


BASE еще хуже. После этого запускаем проверку целиком.

In [ ]:
MAX_EVAL_SAMPLES = 1000

In [ ]:
#Выборка для оценки
eval_df = clean_eval_df.sample(
    min(MAX_EVAL_SAMPLES, len(clean_eval_df)), random_state=42
).reset_index(drop=True)
eval_sci = filter_natural_sciences(eval_df)

print(f'Всего для оценки:      {len(eval_df)}')
print(f'Естественных наук:     {len(eval_sci)}')

Всего для оценки:      1000
Естественных наук:     142


In [ ]:
#Оценка BASE модели
model_base, tok_base = load_model_for_eval(BASE_MODEL_ID, label="BASE")

base_all_df = evaluate_with_logits(model_base, tok_base, eval_df, clean_eval_df, label="BASE_all")
base_sci_df = evaluate_with_logits(model_base, tok_base, eval_sci, clean_eval_df, label="BASE_sci")

base_acc_all = print_results(base_all_df, "BASE все предметы")
base_acc_sci = print_results(base_sci_df, "BASE естественные науки")

base_all_df.to_csv(f"{RESULTS_DIR}/base_all.csv", index=False)
base_sci_df.to_csv(f"{RESULTS_DIR}/base_sci.csv", index=False)

del model_base, tok_base
gc.collect(); torch.cuda.empty_cache()


Загружаем [BASE]: google/gemma-3-4b-it


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Eval [BASE_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

Eval [BASE_sci]:   0%|          | 0/142 [00:00<?, ?it/s]


Результаты: BASE все предметы
Accuracy: 0.2440 (24.4%)  |  n=1000
                                     accuracy   n
domain                                           
college_chemistry                       0.667   3
jurisprudence                           0.600  10
abstract_algebra                        0.500  10
college_physics                         0.500   4
astronomy                               0.500   8
international_law                       0.500  10
high_school_chemistry                   0.455  11
clinical_knowledge                      0.429   7
anatomy                                 0.429   7
high_school_us_history                  0.417  12
public_relations                        0.400   5
marketing                               0.381  21
college_biology                         0.375   8
conceptual_physics                      0.364  11
medical_genetics                        0.333   3
professional_law                        0.325  80
miscellaneous                    

In [ ]:
# Оценка SFT модели
model_sft, tok_sft = load_model_for_eval(SFT_CHECKPOINT, label="SFT")

sft_all_df = evaluate_with_logits(model_sft, tok_sft, eval_df, clean_eval_df, label="SFT_all")
sft_sci_df = evaluate_with_logits(model_sft, tok_sft, eval_sci, clean_eval_df, label="SFT_sci")

sft_acc_all = print_results(sft_all_df, "SFT все предметы")
sft_acc_sci = print_results(sft_sci_df, "SFT естественные науки")

sft_all_df.to_csv(f"{RESULTS_DIR}/sft_all.csv", index=False)
sft_sci_df.to_csv(f"{RESULTS_DIR}/sft_sci.csv", index=False)

del model_sft, tok_sft
gc.collect(); torch.cuda.empty_cache()


Загружаем [SFT]: /content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Eval [SFT_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

Eval [SFT_sci]:   0%|          | 0/142 [00:00<?, ?it/s]


Результаты: SFT все предметы
Accuracy: 0.2450 (24.5%)  |  n=1000
                                     accuracy   n
domain                                           
college_chemistry                       0.667   3
abstract_algebra                        0.600  10
college_biology                         0.500   8
college_physics                         0.500   4
college_medicine                        0.500   4
high_school_us_history                  0.500  12
clinical_knowledge                      0.429   7
high_school_physics                     0.429   7
jurisprudence                           0.400  10
international_law                       0.400  10
public_relations                        0.400   5
college_mathematics                     0.385  13
security_studies                        0.368  19
conceptual_physics                      0.364  11
medical_genetics                        0.333   3
human_sexuality                         0.333   6
world_religions                   

In [ ]:
# Оценка DPO моделей (все beta)
dpo_results = {}

for beta, ckpt_path in sorted(beta_checkpoints.items()):
    model_dpo, tok_dpo = load_model_for_eval(ckpt_path, label=f"DPO beta={beta}")

    dpo_all_df = evaluate_with_logits(model_dpo, tok_dpo, eval_df, clean_eval_df, label=f"DPO_b{beta}_all")
    dpo_sci_df = evaluate_with_logits(model_dpo, tok_dpo, eval_sci, clean_eval_df, label=f"DPO_b{beta}_sci")

    acc_all = print_results(dpo_all_df, f"DPO beta={beta} все предметы")
    acc_sci = print_results(dpo_sci_df, f"DPO beta={beta} естественные науки")

    dpo_all_df.to_csv(f"{RESULTS_DIR}/dpo_beta{str(beta).replace('.','_')}_all.csv", index=False)
    dpo_sci_df.to_csv(f"{RESULTS_DIR}/dpo_beta{str(beta).replace('.','_')}_sci.csv", index=False)

    dpo_results[beta] = {
        "acc_all": acc_all, "acc_sci": acc_sci,
        "all_df": dpo_all_df, "sci_df": dpo_sci_df,
    }

    del model_dpo, tok_dpo
    gc.collect(); torch.cuda.empty_cache()


Загружаем [DPO beta=0.01]: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_01


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Eval [DPO_b0.01_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Eval [DPO_b0.5_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

Eval [DPO_b0.5_sci]:   0%|          | 0/142 [00:00<?, ?it/s]


Результаты: DPO beta=0.5 все предметы
Accuracy: 0.2430 (24.3%)  |  n=1000
                                     accuracy   n
domain                                           
college_chemistry                       0.667   3
abstract_algebra                        0.600  10
college_biology                         0.500   8
college_physics                         0.500   4
college_medicine                        0.500   4
clinical_knowledge                      0.429   7
high_school_physics                     0.429   7
international_law                       0.400  10
jurisprudence                           0.400  10
public_relations                        0.400   5
college_mathematics                     0.385  13
security_studies                        0.368  19
conceptual_physics                      0.364  11
global_facts                            0.333  15
medical_genetics                        0.333   3
human_sexuality                         0.333   6
prehistory               

## 6. Сравнение всех версий

In [ ]:
print("\n" + "="*65)
print("ИТОГОВОЕ СРАВНЕНИЕ")
print("="*65)
print(f"{'Модель':<25} {'Все предметы':>15} {'Естественные науки':>20}")
print("-"*65)
print(f"{'BASE':<25} {base_acc_all*100:>14.1f}% {base_acc_sci*100:>19.1f}%")
print(f"{'SFT':<25} {sft_acc_all*100:>14.1f}% {sft_acc_sci*100:>19.1f}%")
for beta, res in sorted(dpo_results.items()):
    label = f"DPO beta={beta}"
    print(f"{label:<25} {res['acc_all']*100:>14.1f}% {res['acc_sci']*100:>19.1f}%")
print("="*65)


ИТОГОВОЕ СРАВНЕНИЕ
Модель                       Все предметы   Естественные науки
-----------------------------------------------------------------
BASE                                24.4%                29.6%
SFT                                 24.5%                24.6%
DPO beta=0.01                       24.3%                23.2%
DPO beta=0.1                        25.2%                26.1%
DPO beta=0.5                        24.3%                26.1%
